# 1. UAV-Flow datasetexploration

Notebook UAV-Flow dataset and content. UAV-Flow is based on true scene dronedataset, contains FPV ( #) image, instruction and 6-DoF action. 

**target: **
- dataset and format
- drone FPV image
- and action space
- analyzeaction statistics

## 1.1 environment

In [ ]:
import json
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

# datapath
DATA_DIR = Path("/root/autodl-tmp/claude/data/uav_flow_subset")
print(f"data: {DATA_DIR}")
print(f": {len(list(DATA_DIR.iterdir()))}")

## 1.2 data

UAV-Flow dataset by, each is: 

```
uav_flow_subset/
├── 2025-04-02_15-25-53/ # ID ( between)
│ ├── 000000.jpg # # 0 frame FPV image
│ ├── 000001.jpg # # 1 frame
│ ├──... 
│ └── log.json # data + actionlabel
├── 2025-04-03_09-00-12/
│ └──...
└──...
```

In [ ]:
# before 5 
trajectories = sorted(list(DATA_DIR.iterdir()))[:5]
for traj in trajectories:
files = sorted(list(traj.iterdir()))
n_images = len([f for f in files if f.suffix == '.jpg'])
print(f": {traj.name} | frame: {n_images} |: {[f.name for f in files[:3]]}...")

## 1.3 log.json 

each `log.json` contains: 
- `id`: ID
- `instruction`: instruction ( "Go toward the tree on the left side") 
- `raw_logs`: log `[x, y, z, roll, pitch, yaw, timestamp]`
- `preprocessed_logs`: normalization after 6 action `[norm_x, norm_y, norm_z, roll_delta, pitch_delta, yaw_delta]`
- `length`: long (frame) 

In [ ]:
# load log.json
sample_traj = sorted(list(DATA_DIR.iterdir()))[0]
with open(sample_traj / 'log.json', 'r') as f:
log = json.load(f)

print(f" ID: {log['id']}")
print(f"instruction: {log['instruction']}")
print(f" long: {log['length']} frame")
print(f"\nraw_logs: [{len(log['raw_logs'])} x {len(log['raw_logs'][0])}]")
print(f"raw_logs #frame: {log['raw_logs'][0]}")
print(f": [x, y, z, roll, pitch, yaw, timestamp]")
print(f"\npreprocessed_logs: [{len(log['preprocessed_logs'])} x {len(log['preprocessed_logs'][0])}]")
print(f"preprocessed_logs #frame: {log['preprocessed_logs'][0]}")
print(f": [norm_x, norm_y, norm_z, roll_delta, pitch_delta, yaw_delta]")

## 1.4 FPV image

In [ ]:
# keyframe
images = sorted(list(sample_traj.glob('*.jpg')))
n_frames = len(images)

# sampling 6 frame
indices = np.linspace(0, n_frames - 1, 6, dtype=int)
fig, axes = plt.subplots(1, 6, figsize=(18, 3))
fig.suptitle(f": {log['id']}\ninstruction: {log['instruction']}", fontsize=13)

for ax, idx in zip(axes, indices):
img = Image.open(images[idx])
ax.imshow(img)
ax.set_title(f"Frame {idx}")
ax.axis('off')

plt.tight_layout()
plt.show()

## 1.5 (3D)

from `raw_logs` in x, y, z, drone 3D flight path. 

In [ ]:
raw = np.array(log['raw_logs'])
x, y, z = raw[:, 0], raw[:, 1], raw[:, 2]

# 
x = x - x[0]
y = y - y[0]
z = z - z[0]

fig = plt.figure(figsize=(10, 7))
ax = fig.add_subplot(111, projection='3d')

# 
ax.plot(x, y, z, 'b-o', markersize=3, linewidth=1.5, label='Flight Path')
ax.scatter(x[0], y[0], z[0], color='green', s=100, marker='^', label='Start')
ax.scatter(x[-1], y[-1], z[-1], color='red', s=100, marker='*', label='End')

ax.set_xlabel('X (m)')
ax.set_ylabel('Y (m)')
ax.set_zlabel('Z (m)')
ax.set_title(f'3D Flight Trajectory\nInstruction: {log["instruction"]}')
ax.legend()
plt.tight_layout()
plt.show()

## 1.6 action spaceanalyze

UAV-Flow action is 6-DoF: `[x, y, z, roll, pitch, yaw]`

in OpenVLA-UAV fine-tuning in, use 4: **`[x, y, z, yaw]`** (roll and pitch for short distance small). 

below analyze has in actionvalue. 

In [ ]:
# has preprocessed_logs
all_actions = []
all_instructions = []

for traj_dir in sorted(DATA_DIR.iterdir()):
log_path = traj_dir / 'log.json'
if not log_path.exists():
continue
with open(log_path, 'r') as f:
log_data = json.load(f)
actions = np.array(log_data['preprocessed_logs'])
all_actions.append(actions)
all_instructions.append(log_data.get('instruction', ''))

all_actions = np.concatenate(all_actions, axis=0)
print(f"action: {len(all_actions)}")
print(f"action: {all_actions.shape[1]}")
print(f"instruction: {len(set(all_instructions))}")

In [ ]:
# actiongraph
dim_names = ['X ( before after)', 'Y ()', 'Z ( above below)', 'Roll', 'Pitch', 'Yaw ()']
used_dims = [0, 1, 2, 5] # use 4: x, y, z, yaw

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.flatten()

for i, dim_idx in enumerate(used_dims):
data = all_actions[:, dim_idx]
axes[i].hist(data, bins=50, color='steelblue', alpha=0.7, edgecolor='white')
axes[i].axvline(np.percentile(data, 1), color='red', linestyle='--', label='1st/99th percentile')
axes[i].axvline(np.percentile(data, 99), color='red', linestyle='--')
axes[i].axvline(np.mean(data), color='orange', linestyle='-', label=f'Mean={np.mean(data):.4f}')
axes[i].set_title(f'{dim_names[dim_idx]}')
axes[i].legend(fontsize=9)

fig.suptitle('actionvalue (training use 4)', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# actionstatistics need 
print("actionstatistics ( use training 4: x, y, z, yaw):")
print("-" * 60)
for i, dim_idx in enumerate(used_dims):
data = all_actions[:, dim_idx]
print(f"{dim_names[dim_idx]:>12s}: mean={np.mean(data):+.4f} std={np.std(data):.4f} "
f"1%={np.percentile(data, 1):+.4f} 99%={np.percentile(data, 99):+.4f}")

print("\n** 1st/99th use Action Tokenizer edge **")

## 1.7 instructionstatistics

In [ ]:
from collections import Counter

# key
words = []
for inst in all_instructions:
words.extend(inst.lower().split())

word_freq = Counter(words)
top_words = word_freq.most_common(15)

print(" before 15 high:")
for word, count in top_words:
print(f" {word:>12s}: {count}")

print(f"\nexampleinstruction:")
for inst in list(set(all_instructions))[:8]:
print(f" - {inst}")

## 1.8 many for 

In [ ]:
# 4, image + 
np.random.seed(42)
sample_dirs = np.random.choice(sorted(list(DATA_DIR.iterdir())), size=4, replace=False)

fig, axes = plt.subplots(2, 4, figsize=(16, 8))

for col, traj_dir in enumerate(sample_dirs):
with open(traj_dir / 'log.json', 'r') as f:
log_data = json.load(f)

# above: in between frameimage
imgs = sorted(list(traj_dir.glob('*.jpg')))
mid_img = Image.open(imgs[len(imgs)//2])
axes[0, col].imshow(mid_img)
axes[0, col].set_title(log_data['instruction'][:30] + '...', fontsize=9)
axes[0, col].axis('off')

# below: 2D (graph)
raw = np.array(log_data['raw_logs'])
rx, ry = raw[:, 0] - raw[0, 0], raw[:, 1] - raw[0, 1]
axes[1, col].plot(rx, ry, 'b-o', markersize=2)
axes[1, col].scatter(rx[0], ry[0], color='green', s=50, zorder=5, marker='^')
axes[1, col].scatter(rx[-1], ry[-1], color='red', s=50, zorder=5, marker='*')
axes[1, col].set_xlabel('X (m)')
axes[1, col].set_ylabel('Y (m)')
axes[1, col].set_aspect('equal')
axes[1, col].grid(True, alpha=0.3)

axes[0, 0].set_ylabel('FPV Image', fontsize=12)
axes[1, 0].set_ylabel('Top-down Trajectory', fontsize=12)
plt.suptitle('UAV-Flow: image + ', fontsize=14)
plt.tight_layout()
plt.show()

## small 

- UAV-Flow datasetcontains true scene drone
- each FPV image + instruction + 6-DoF action
- training use 4 action `[x, y, z, yaw]`, through 1st/99th normalization
- action need in in before toward (x), yaw control toward 

** below:** in Notebook 02 in, OpenVLA image and instructionconvertaction. 